# 320_Fine_Tuning_Custom_Datasets

In [2]:
!pip install transformers

     |████████████████████████████████| 2.6 MB 9.4 MB/s 
     |████████████████████████████████| 3.3 MB 52.5 MB/s 
     |████████████████████████████████| 636 kB 35.6 MB/s 
     |████████████████████████████████| 895 kB 49.1 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


In [3]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import torch.nn.functional as F
import tensorflow as tf
import pandas as pd

In [4]:
DATA_TRAIN_PATH = tf.keras.utils.get_file("ratings_train.txt", 
                                "https://raw.github.com/ironmanciti/NLP_lecture/master/data/naver_movie/ratings_train.txt")
DATA_TEST_PATH = tf.keras.utils.get_file("ratings_test.txt", 
                                 "https://raw.github.com/ironmanciti/NLP_lecture/master/data/naver_movie/ratings_test.txt")

4956160/4943336 [==============================] - 0s 0us/step


In [5]:
train_data = pd.read_csv(DATA_TRAIN_PATH, delimiter='\t')
print(train_data.shape)
train_data.head()

(150000, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
train_data.dropna(inplace=True)
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB


In [7]:
test_data = pd.read_csv(DATA_TEST_PATH, delimiter='\t')
print(test_data.shape)
test_data.head()

(50000, 3)


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [8]:
test_data.dropna(inplace=True)
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 49997 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49997 non-null  int64 
 1   document  49997 non-null  object
 2   label     49997 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ MB


In [9]:
df_train = train_data.sample(n=5000, random_state=1)
df_test = test_data.sample(n=500, random_state=1)
print(df_train.shape)
print(df_test.shape)

(5000, 3)
(500, 3)


In [10]:
df_train['label'].value_counts()

0    2520
1    2480
Name: label, dtype: int64

In [11]:
X_train = df_train['document'].values.tolist()
y_train = df_train['label'].values

X_test = df_test['document'].values.tolist()
y_test = df_test['label'].values

### 1. Call the pre-trained model
### 2. Call the tokenizer
이제 토큰화를 처리해 보겠습니다. 우리는 사전 훈련된 DistilBert를 사용하여 분류기를 훈련할 것이므로 DistilBert 토크나이저를 사용합시다.

In [12]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
# tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

Downloading:   0%|          | 0.00/996k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/625 [00:00<?, ?B/s]

In [13]:
train_encodings = tokenizer(X_train, truncation=True, padding=True)
test_encodings = tokenizer(X_test, truncation=True, padding=True)

In [14]:
train_encodings.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

### Convert encodings to Tensors

이제 레이블과 인코딩을 Dataset 개체로 변환해 보겠습니다. PyTorch에서 이것은 `torch.utils.data.Dataset` 객체를 서브클래싱하고 `__len__` 및 `__getitem__`을 구현하여 수행됩니다. TensorFlow에서 입력 인코딩과 레이블을 `from_tensor_slices` 생성자 메서드에 전달합니다. 
배치 인코딩의 각 키가 우리가 훈련할 모델의 `DistilBertForSequenceClassification.forward` 메소드의 명명된 매개변수에 해당하도록 쉽게 배치할 수 있습니다.

In [54]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = IMDbDataset(train_encodings, y_train)
test_dataset = IMDbDataset(test_encodings, y_test)

이제 데이터 세트가 준비되었으므로 🤗 `Trainer`/`TFTrainer` 또는 기본 PyTorch/TensorFlow를 사용하여 모델을 미세 조정할 수 있습니다. [training](https://huggingface.co/transformers/training.html)을 참조하세요.

- Training warmup steps :  

    - 이는 일반적으로 설정된 수의 훈련 단계(워밍업 단계)에 대해 매우 낮은 학습률을 사용한다는 것을 의미합니다. 워밍업 단계 후에 "일반" 학습률 또는 학습률 스케줄러를 사용합니다. 또한 워밍업 단계 수에 따라 학습률을 점진적으로 높일 수 있습니다.

- weight_decay : 가중치 감쇠. L2 regularization

In [16]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=2,              # total number of training epochs
    per_device_train_batch_size=8,  # batch size per device during training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
)

### Training

In [17]:
model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased')

trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset             # evaluation dataset
)

trainer.train()

Downloading:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model ch

Step,Training Loss
10,0.693500
20,0.688200
30,0.692700
40,0.688400
50,0.685700
60,0.702800
70,0.683700
80,0.659500
90,0.689400
100,0.690700


Saving model checkpoint to ./results/checkpoint-500
Configuration saved in ./results/checkpoint-500/config.json
Model weights saved in ./results/checkpoint-500/pytorch_model.bin
Saving model checkpoint to ./results/checkpoint-1000
Configuration saved in ./results/checkpoint-1000/config.json
Model weights saved in ./results/checkpoint-1000/pytorch_model.bin


Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=1250, training_loss=0.5410470478057862, metrics={'train_runtime': 190.3097, 'train_samples_per_second': 52.546, 'train_steps_per_second': 6.568, 'total_flos': 616666536000000.0, 'train_loss': 0.5410470478057862, 'epoch': 2.0})

In [18]:
trainer.evaluate(test_dataset)

***** Running Evaluation *****
  Num examples = 500
  Batch size = 16


{'epoch': 2.0,
 'eval_loss': 0.4788724482059479,
 'eval_runtime': 1.6588,
 'eval_samples_per_second': 301.426,
 'eval_steps_per_second': 19.291}

In [19]:
prediction = trainer.predict(test_dataset)
prediction

***** Running Prediction *****
  Num examples = 500
  Batch size = 16


PredictionOutput(predictions=array([[ 0.8485763 , -0.7297784 ],
       [ 1.1591598 , -0.961922  ],
       [ 1.2761818 , -1.09666   ],
       [ 1.6204393 , -1.5558041 ],
       [-0.42283344,  0.5847641 ],
       [ 1.6240362 , -1.5701936 ],
       [-0.15356077,  0.2954778 ],
       [ 0.9486881 , -0.76435316],
       [-1.6589617 ,  1.6999959 ],
       [ 1.0179737 , -0.83315307],
       [-1.667927  ,  1.7211415 ],
       [ 0.00684868,  0.15477955],
       [ 1.563712  , -1.460004  ],
       [ 0.9829624 , -0.8271281 ],
       [-0.08028116,  0.20641445],
       [ 1.2602435 , -1.0579846 ],
       [ 1.6224183 , -1.5728186 ],
       [-1.342532  ,  1.4737881 ],
       [ 1.5970389 , -1.531518  ],
       [ 0.32985806, -0.17224623],
       [ 1.6145662 , -1.5677181 ],
       [ 1.3626186 , -1.1987771 ],
       [-0.03956339,  0.20842141],
       [ 0.00471243,  0.16170405],
       [ 1.5247077 , -1.4382635 ],
       [ 1.1031789 , -0.93138635],
       [ 0.61438775, -0.46630186],
       [-1.5085644 ,  1.53

In [20]:
y_logit = torch.tensor(prediction[0])
y_logit

tensor([[ 0.8486, -0.7298],
        [ 1.1592, -0.9619],
        [ 1.2762, -1.0967],
        [ 1.6204, -1.5558],
        [-0.4228,  0.5848],
        [ 1.6240, -1.5702],
        [-0.1536,  0.2955],
        [ 0.9487, -0.7644],
        [-1.6590,  1.7000],
        [ 1.0180, -0.8332],
        [-1.6679,  1.7211],
        [ 0.0068,  0.1548],
        [ 1.5637, -1.4600],
        [ 0.9830, -0.8271],
        [-0.0803,  0.2064],
        [ 1.2602, -1.0580],
        [ 1.6224, -1.5728],
        [-1.3425,  1.4738],
        [ 1.5970, -1.5315],
        [ 0.3299, -0.1722],
        [ 1.6146, -1.5677],
        [ 1.3626, -1.1988],
        [-0.0396,  0.2084],
        [ 0.0047,  0.1617],
        [ 1.5247, -1.4383],
        [ 1.1032, -0.9314],
        [ 0.6144, -0.4663],
        [-1.5086,  1.5305],
        [ 0.8880, -0.7447],
        [ 0.8468, -0.7095],
        [ 1.6078, -1.5452],
        [ 1.5947, -1.5525],
        [-1.5035,  1.5136],
        [-1.3732,  1.3930],
        [ 1.5993, -1.5574],
        [ 1.6133, -1

In [21]:
y_pred = F.softmax(y_logit).argmax(axis=1).numpy()
y_pred

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  """Entry point for launching an IPython kernel.


array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0,
       1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1,

In [22]:
from sklearn.metrics import confusion_matrix, accuracy_score

print(accuracy_score(y_test, y_pred))

cm=confusion_matrix(y_test, y_pred)
cm

0.79


array([[206,  52],
       [ 53, 189]])

In [23]:
x = '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나'
tokenized = tokenizer(x, truncation=True, padding=True)
tokenized

{'input_ids': [101, 100, 119, 119, 119, 9928, 58823, 30005, 11664, 9757, 118823, 30858, 18227, 119219, 119, 119, 119, 119, 9580, 41605, 25486, 12310, 20626, 23466, 8843, 118986, 12508, 9523, 17196, 16439, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [24]:
tokenizer.decode(tokenized['input_ids'])

'[CLS] [UNK]... 포스터보고 초딩영화줄.... 오버연기조차 가볍지 않구나 [SEP]'

In [66]:
input_ids = torch.tensor(tokenized['input_ids'])
attention_mask = torch.tensor(tokenized['attention_mask'])

In [67]:
model(input_ids, attention_mask)

ValueError: ignored